# 07C – LIME Local Explainability

Enterprise notebook for explaining individual bankruptcy predictions using **LIME**.

## Business Objective

Use LIME (Local Interpretable Model-agnostic Explanations) to explain why the model predicted bankruptcy risk for a single company and compare these insights with SHAP.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
MODEL_PATH='production_bankruptcy_model.joblib'
DATA_PATH='american_bankruptcy.csv'
INSTANCE_INDEX=0

model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y=df['status_label'].map({'alive':0,'failed':1})
    X=df.drop(columns=[c for c in ['status_label','company_name'] if c in df.columns])
elif 'target' in df.columns:
    y=df['target']
    X=df.drop(columns=[c for c in ['target','company_name'] if c in df.columns])
else:
    raise ValueError('Target column not found')

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)

instance=X_test.iloc[INSTANCE_INDEX]

In [ ]:
explainer=LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Healthy','Bankrupt'],
    mode='classification',
    discretize_continuous=True
)

exp=explainer.explain_instance(
    instance.values,
    model.predict_proba,
    num_features=15
)

In [ ]:
# Save interactive explanation
exp.save_to_file('lime_explanation.html')

weights=pd.DataFrame(
    exp.as_list(),
    columns=['Feature','Contribution']
)

weights.to_csv('lime_feature_contributions.csv',index=False)
weights

## Business Interpretation

- Positive contributions increase the predicted bankruptcy probability.
- Negative contributions reduce the predicted bankruptcy probability.
- LIME provides an intuitive local explanation for one prediction, making it valuable for model validation and stakeholder communication.

## Deliverables

- `lime_explanation.html`
- `lime_feature_contributions.csv`

### Portfolio Value
This notebook demonstrates model-agnostic local explainability and complements SHAP-based explanations.